# 07 — Visualization and quality control

This notebook covers four complementary QC tasks:

1. view matched H&E/IHC patches;
2. draw tissue-selection grid maps;
3. generate multi-marker IHC overlays; and
4. export a high-resolution H&E/CD8 representative figure.

Install `.[viz]`; overlay masking also needs OpenCV.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    '''Find the RocqiPath repository whether Jupyter starts at root or how_to_use.'''
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "src" / "rocqipath"
        ).is_dir():
            return candidate
    raise FileNotFoundError(
        "RocqiPath repository not found. Start Jupyter inside the cloned repository."
    )


PROJECT_ROOT = find_project_root()
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

DATA_ROOT = PROJECT_ROOT / "data"
RESULTS_ROOT = PROJECT_ROOT / "results"

print(f"Project : {PROJECT_ROOT}")
print(f"Data    : {DATA_ROOT}")
print(f"Results : {RESULTS_ROOT}")


In [ ]:
PATCH_CASE_DIR = RESULTS_ROOT / "patch_extraction" / "Sample_0001_CD8"
OVERLAY_INPUT_ROOT = DATA_ROOT / "overlay_patches"
OUTPUT_ROOT = RESULTS_ROOT

HE_SLIDE = DATA_ROOT / "wsi" / "Sample_0001_he.svs"
ALIGNED_CD8_SLIDE = (
    RESULTS_ROOT
    / "alignment"
    / "Sample_0001_cd8"
    / "Sample_0001_cd8_aligned_moving.ome.tiff"
)

TARGET_MAGNIFICATION = 20.0
HE_SOURCE_MAGNIFICATION = None
CD8_SOURCE_MAGNIFICATION = None
RUN_REPRESENTATIVE_FIGURE = False
RUN_OVERLAYS = False


## View matched patch pairs

`view_pairs` reads the case manifest first, so custom channel names remain
paired correctly. Use a small random sample during routine QC and inspect
every failure-prone case separately.


In [ ]:
from rocqipath.visualization import view_pairs

if PATCH_CASE_DIR.is_dir():
    view_pairs(str(PATCH_CASE_DIR), num_to_show=5)
else:
    print(f"Patch case not found: {PATCH_CASE_DIR}")


## Grid-map smoke test

The public `plot_selector_map` function accepts any PIL thumbnail and a set
of row-major valid grid IDs. In a real pipeline, the registrar produces
both values.


In [ ]:
from PIL import Image, ImageDraw

from rocqipath.visualization import plot_selector_map

thumb = Image.new("RGB", (800, 600), (245, 245, 245))
draw = ImageDraw.Draw(thumb)
draw.ellipse((80, 90, 710, 520), fill=(205, 145, 180))

rows = cols = 6
valid_ids = {7, 8, 9, 10, 13, 14, 15, 16, 19, 20, 21, 22, 26, 27}
grid_output = OUTPUT_ROOT / "visualization" / "grid_map_demo.png"
grid_output.parent.mkdir(parents=True, exist_ok=True)

plot_selector_map(
    thumb,
    valid_ids,
    rows,
    cols,
    output_path=str(grid_output),
    show=True,
)
print(grid_output)


## Prepare multi-marker overlay input

`process_ihc_overlay` expects one folder per marker and exactly matching
filenames across markers:

```text
overlay_patches/
└── Sample_0001/
    ├── CD8/patch_000001.png
    ├── CD31/patch_000001.png
    └── CAIX/patch_000001.png
```

When patches came from separate flat RocqiPath case folders, the helper
below copies them into this comparison layout using the six-digit patch ID.


In [ ]:
import re
import shutil


def organize_marker_cases_for_overlay(
    marker_sources: dict[str, tuple[Path, str]],
    destination_case: Path,
) -> dict[str, int]:
    '''Copy one selected channel per marker to common patch-ID filenames.'''
    counts: dict[str, int] = {}
    for marker, (source_dir, channel_token) in marker_sources.items():
        marker_out = destination_case / marker
        marker_out.mkdir(parents=True, exist_ok=True)
        count = 0
        pattern = re.compile(
            rf"_{re.escape(channel_token)}_patch_(\d{{6}})",
            re.IGNORECASE,
        )
        for source in sorted(source_dir.glob("*.png")):
            match = pattern.search(source.name)
            if match is None:
                continue
            destination = marker_out / f"patch_{match.group(1)}.png"
            if not destination.exists():
                shutil.copy2(source, destination)
            count += 1
        counts[marker] = count
    return counts


# Example:
# counts = organize_marker_cases_for_overlay(
#     {
#         "CD8": (
#             RESULTS_ROOT / "patch_extraction" / "Sample_0001_CD8",
#             "cd8",
#         ),
#         "CD31": (
#             RESULTS_ROOT / "patch_extraction" / "Sample_0001_CD31",
#             "cd31",
#         ),
#         "CAIX": (
#             RESULTS_ROOT / "patch_extraction" / "Sample_0001_CAIX",
#             "caix",
#         ),
#     },
#     OVERLAY_INPUT_ROOT / "Sample_0001",
# )
# print(counts)


In [ ]:
from rocqipath.visualization import (
    IHCOverlayConfig,
    MarkerProfile,
    OverlayCombo,
    process_ihc_overlay,
)

overlay_cfg = IHCOverlayConfig(
    markers={
        "CD8": MarkerProfile(
            color=(220, 30, 45),
            label="CD8",
            hue_range=(5, 20),
            sat_min=30,
        ),
        "CD31": MarkerProfile(
            color=(20, 135, 210),
            label="CD31",
            hue_range=(5, 20),
            sat_min=30,
        ),
        "CAIX": MarkerProfile(
            color=(250, 195, 25),
            label="CAIX",
            hue_range=(5, 20),
            sat_min=30,
        ),
    },
    combinations=[
        OverlayCombo(base="CD31", overlays=["CD8", "CAIX"]),
    ],
    base_marker="CD31",
    base_render_mode="original",
    plot_mode="both",
    save_dir=str(OUTPUT_ROOT),
    dpi=300,
    patches_per_case=10,
    skip_existing=True,
    max_workers=2,
    show_plot=False,
)

if RUN_OVERLAYS:
    overlay_results = process_ihc_overlay(
        data_in=str(OVERLAY_INPUT_ROOT),
        cfg=overlay_cfg,
    )
    print(overlay_results)
else:
    print("Set RUN_OVERLAYS=True after organizing marker folders.")


## High-resolution representative H&E/CD8 figure

Choose target-grid coordinates after alignment. Both panels are read from
the same `(x, y, width, height)` at the same physical magnification. Save
PNG for slides and PDF for vector text in publications.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from rocqipath.core import SlideReader


def save_representative_pair(
    he_path: Path,
    cd8_path: Path,
    output_stem: Path,
    *,
    location: tuple[int, int],
    size: tuple[int, int] = (1400, 1000),
    target_magnification: float = 20.0,
    he_source_magnification: float | None = None,
    cd8_source_magnification: float | None = None,
    dpi: int = 600,
) -> tuple[Path, Path]:
    '''Read matching target-grid fields and save PNG/PDF panels.'''
    with SlideReader(str(he_path)) as he_reader, SlideReader(str(cd8_path)) as cd8_reader:
        he_reader.configure_magnification(
            target_magnification, he_source_magnification
        )
        cd8_reader.configure_magnification(
            target_magnification, cd8_source_magnification
        )
        if he_reader.target_dimensions != cd8_reader.target_dimensions:
            raise ValueError(
                "H&E and aligned CD8 dimensions differ at target magnification: "
                f"{he_reader.target_dimensions} vs {cd8_reader.target_dimensions}"
            )
        he = np.array(
            he_reader.read_at_magnification(location, size).convert("RGB")
        )
        cd8 = np.array(
            cd8_reader.read_at_magnification(location, size).convert("RGB")
        )

    fig, axes = plt.subplots(1, 2, figsize=(12, 5.2))
    for ax, image, title in zip(axes, (he, cd8), ("H&E", "CD8 IHC")):
        ax.imshow(image)
        ax.set_title(title, fontsize=16, fontweight="bold")
        ax.axis("off")
    fig.suptitle(
        f"Representative aligned tissue at {target_magnification:g}x",
        fontsize=17,
    )
    plt.tight_layout()

    output_stem.parent.mkdir(parents=True, exist_ok=True)
    png_path = output_stem.with_suffix(".png")
    pdf_path = output_stem.with_suffix(".pdf")
    fig.savefig(png_path, dpi=dpi, bbox_inches="tight", facecolor="white")
    fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return png_path, pdf_path


if RUN_REPRESENTATIVE_FIGURE:
    if not HE_SLIDE.is_file() or not ALIGNED_CD8_SLIDE.is_file():
        raise FileNotFoundError("Edit HE_SLIDE and ALIGNED_CD8_SLIDE.")
    figure_paths = save_representative_pair(
        HE_SLIDE,
        ALIGNED_CD8_SLIDE,
        OUTPUT_ROOT / "visualization" / "Sample_0001_he_cd8_representative",
        location=(5000, 5000),
        size=(1400, 1000),
        target_magnification=TARGET_MAGNIFICATION,
        he_source_magnification=HE_SOURCE_MAGNIFICATION,
        cd8_source_magnification=CD8_SOURCE_MAGNIFICATION,
        dpi=600,
    )
    print(*figure_paths, sep="\n")
else:
    print("Set RUN_REPRESENTATIVE_FIGURE=True after selecting coordinates.")


## QC checklist

- Inspect low-texture and tissue-edge regions, not only dense tissue.
- Confirm the same anatomic structures appear at the same target-grid
  coordinates.
- Check overlay mask specificity on weak stain, hematoxylin, red cells,
  pigment, folds, and background.
- Record the selected representative coordinates and magnification.
- Use 300 DPI for routine QC and 600 DPI for final raster figures.
- Prefer PDF/SVG for text and annotations when the target venue accepts it.
